# OmniPhyto: docking workflow (Colab)

**Run the cells top to bottom.** Every step saves its results to `omniphyto_run/` inside your Drive project folder, so if Colab disconnects you can re-run Step 1 and carry on: finished work is skipped.

Things to know before you start
- **Vina uses the CPU only.** A GPU runtime does not make it faster, so keep the runtime on **CPU** and save your GPU quota. More CPU cores is what speeds it up.
- **Time guard.** Step 1 sets a session time budget. Docking checks it after every compound and stops cleanly, with partial results saved, before the budget is spent. Colab does not report remaining compute units to code, so the guard is wall-clock based.
- Changes from the original brief, each for a scientific reason: the CYP enzymes keep their heme (docking without it is meaningless); hERG keeps chains A-D (the drug site sits between the four subunits); the DPP-4 box is the 22 Å cube you asked for.

## STEP 1: Environment, Drive, settings

In [ ]:
# STEP 1A: mount Drive, settings, time budget, helpers
import os, sys, time, glob, json, math, shutil, subprocess, csv, threading, re, warnings
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
warnings.filterwarnings('ignore')

IN_COLAB = False
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except Exception as e:
    print('Not in Colab (or Drive mount failed):', e)

# ===================== SETTINGS: the only things you normally edit =====================
PROJECT_DIR        = ''     # '' = auto-find the Drive folder that holds compounds_passed.csv
DRIVE_FOLDER_ID    = '1vdNpnnkreyKU64VkDw8RquS4SJOp24v9'   # your shared project folder
SESSION_BUDGET_MIN = 300    # stop docking after this many minutes in total (free Colab can cut you off earlier)
SAFETY_MARGIN_MIN  = 20     # always keep this long free for saving and the report
EXHAUSTIVENESS     = 0      # 0 = auto-pick from a timing test; or set 4 / 8 / 16 yourself
ANTI_EXHAUSTIVENESS = 16    # search effort for the 3 safety receptors (only 10 compounds each)
MAX_LIGANDS        = 0      # 0 = all; set e.g. 12 for a quick dry run
DEDUP_STEREOISOMERS = True  # dock one stereoisomer per parent structure (saves roughly 20% of the time)
TOP_N              = 10     # how many therapeutic hits go to safety docking and the report
RANK_BY            = 'vina_score_kcal'   # or 'ligand_efficiency' (corrects for molecule size)
SYNC_EVERY         = 20     # copy results to Drive every N dockings
# =======================================================================================

T0 = time.time()
class Budget:
    def __init__(self, total_min, margin_min): self.total, self.margin = total_min, margin_min
    def elapsed(self): return (time.time() - T0) / 60
    def remaining(self): return self.total - self.elapsed()
    def should_stop(self): return self.remaining() <= self.margin
BUDGET = Budget(SESSION_BUDGET_MIN, SAFETY_MARGIN_MIN)

def find_project():
    roots = ['/content/drive/MyDrive', '/content/drive/Shareddrives']
    for root in roots:
        if not os.path.isdir(root): continue
        base = root.rstrip('/').count('/')
        for dp, dn, fn in os.walk(root, followlinks=True):
            dn[:] = [d for d in dn if not d.startswith('.')]
            if dp.count('/') - base >= 4: dn[:] = []
            if 'compounds_passed.csv' in fn or 'compounds_scored.csv' in fn: return dp
    return ''

if not PROJECT_DIR: PROJECT_DIR = os.environ.get('OMNI_PROJECT', '') or find_project()
if not PROJECT_DIR:
    raise SystemExit('Could not find compounds_passed.csv in your Drive.\n'
                     f'Open https://drive.google.com/drive/folders/{DRIVE_FOLDER_ID} , right-click the folder > Organize > Add shortcut > My Drive, '
                     'then re-run this cell (or paste the folder path into PROJECT_DIR).')
PROJECT = Path(PROJECT_DIR)
OUT  = PROJECT / 'omniphyto_run'                                   # saved copy on Drive
WORK = Path(os.environ.get('OMNI_WORK', '/content/omniphyto'))     # fast local working copy
for d in ('receptors', 'ligands', 'validation', 'results', 'data_raw', 'bin'): (WORK / d).mkdir(parents=True, exist_ok=True)
OUT.mkdir(exist_ok=True)

SYNC_DIRS = ['receptors', 'ligands', 'validation', 'results', 'data_raw']
def sync(dirs=SYNC_DIRS):
    """local -> Drive"""
    for d in dirs:
        if (WORK / d).exists(): shutil.copytree(WORK / d, OUT / d, dirs_exist_ok=True)
def restore():
    """Drive -> local, after a disconnect"""
    for d in SYNC_DIRS:
        if (OUT / d).exists(): shutil.copytree(OUT / d, WORK / d, dirs_exist_ok=True)

def status(tag=''):
    msg = f'[{tag}] elapsed {BUDGET.elapsed():.0f} min | budget left {BUDGET.remaining():.0f} min'
    try:
        import psutil
        msg += f" | RAM {psutil.virtual_memory().percent:.0f}% | disk free {shutil.disk_usage(str(WORK)).free/1e9:.1f} GB | CPU {psutil.cpu_percent():.0f}%"
    except Exception: pass
    print(msg)

restore()
print('Project folder :', PROJECT)
print('Results saved to:', OUT)
status('start')

In [ ]:
# STEP 1B: install everything (about 2-4 minutes). If Colab asks you to restart the session afterwards, do it, then run from STEP 1A again.
!pip -q install "meeko==0.8.0" rdkit gemmi scipy spyrmsd pandas pubchempy pdbfixer openmm openbabel-wheel lxml matplotlib psutil
!pip -q install --no-deps plip
!pip -q install dimorphite-dl
# AutoDock Vina 1.2.5: pre-compiled Linux program (more reliable than pip install vina)
!test -x /content/omniphyto/bin/vina || (wget -q -O /content/omniphyto/bin/vina https://github.com/ccsb-scripps/AutoDock-Vina/releases/download/v1.2.5/vina_1.2.5_linux_x86_64 && chmod +x /content/omniphyto/bin/vina)

In [ ]:
# STEP 1C: check the install and the machine
VINA = shutil.which('vina') or str(WORK / 'bin' / 'vina')
if not (os.path.exists(VINA) and os.access(VINA, os.X_OK)):
    VINA = '/content/omniphyto/bin/vina'
print('Vina :', subprocess.run([VINA, '--version'], capture_output=True, text=True).stdout.strip())
import rdkit, meeko, gemmi, pandas, spyrmsd
print('RDKit', rdkit.__version__, '| Meeko', meeko.__version__, '| gemmi', gemmi.__version__, '| spyrmsd', spyrmsd.__version__)
try:
    from openbabel import pybel; print('Open Babel OK')
except Exception as e: print('Open Babel problem:', e)
try:
    import plip; print('PLIP OK')
except Exception as e: print('PLIP problem (Step 6 will be skipped):', e)

NCPU = os.cpu_count() or 2
WORKERS = max(1, NCPU)          # one Vina process per core
print(f'CPU cores: {NCPU} -> docking {WORKERS} compounds at a time')
if subprocess.run('nvidia-smi', shell=True, capture_output=True).returncode == 0:
    print('NOTE: a GPU is attached but Vina cannot use it. Switch Runtime > Change runtime type > CPU to save your GPU quota.')
status('install ok')

## STEP 2: Download and prepare the four receptors
DPP-4 is the target; hERG, CYP3A4 and CYP2D6 are the safety receptors. Each one is prepared inside its own try/except, so one failure never stops the rest. The table at the end says which receptors are ready.

In [ ]:
# STEP 2: receptors
import numpy as np, pandas as pd, urllib.request
from pdbfixer import PDBFixer
from openmm.app import PDBFile

RAW, REC = WORK / 'data_raw', WORK / 'receptors'

RECEPTORS = {
    # name: settings.  chains = protein chains kept. ligand/ligand_chain = co-crystal ligand that defines the box.
    'dpp4':   dict(pdb='3D4L', chains='A',    ligand='605', ligand_chain='A', engine='meeko', box=(22, 22, 22)),
    'herg':   dict(pdb='8ZYO', chains='ABCD', ligand='XB7', ligand_chain='D', engine='meeko'),
    'cyp3a4': dict(pdb='2V0M', chains='A',    ligand='KLN', ligand_chain='A', keep_het='HEM', engine='obabel'),   # ketoconazole; 2 copies per chain, the one nearest the heme is used
    'cyp2d6': dict(pdb='3TDA', chains='A',    ligand='PN0', ligand_chain='A', keep_het='HEM', engine='obabel'),
}

def fetch_pdb(pid):
    dst = RAW / f'{pid}.pdb'
    if dst.exists() and dst.stat().st_size > 1000: return dst          # already downloaded (or uploaded by you)
    last = None
    for ext in ('pdb', 'cif'):
        try:
            data = urllib.request.urlopen(f'https://files.rcsb.org/download/{pid}.{ext}', timeout=60).read()
        except Exception as e:
            last = e; continue
        if ext == 'pdb': dst.write_bytes(data)
        else:
            tmp = RAW / f'{pid}.cif'; tmp.write_bytes(data)
            st = gemmi.read_structure(str(tmp)); st.setup_entities(); st.write_pdb(str(dst))
        return dst
    raise RuntimeError(f'could not download {pid}: {last}. Upload {pid}.pdb to {RAW} and re-run.')

def obabel_rigid(in_pdb, out_pdbqt):
    """rigid receptor PDBQT with Gasteiger charges via Open Babel (used for heme enzymes)"""
    try:
        from openbabel import pybel
        mol = next(pybel.readfile('pdb', str(in_pdb)))
        mol.write('pdbqt', str(out_pdbqt), overwrite=True, opt={'r': None})
    except Exception:
        subprocess.run(['obabel', str(in_pdb), '-O', str(out_pdbqt), '-xr'], check=True, capture_output=True)
    if not Path(out_pdbqt).exists(): raise RuntimeError('Open Babel wrote nothing')

def prep_receptor(name, s):
    pdb = fetch_pdb(s['pdb'])
    chains, code, lchain = s['chains'], s.get('ligand', '').upper(), s.get('ligand_chain', '')
    keep = {x.strip().upper() for x in s.get('keep_het', '').split(',') if x.strip()}
    prot, het, lig = [], [], []
    for line in open(pdb):
        line = line.rstrip('\r\n').ljust(80) + '\n'            # real PDB files have short lines (END, TER)
        rec, ch, res, alt = line[:6], line[21], line[17:20].strip().upper(), line[16]
        if rec == 'ATOM  ' and ch in chains and alt in ' A': prot.append(line)
        elif rec == 'HETATM' and alt in ' A':
            if code and res == code and ch == lchain: lig.append(line)
            elif res in keep and ch in chains: het.append(line)
    if not prot: raise RuntimeError('no protein atoms: check chains')
    copies = {}
    for l in lig: copies.setdefault(l[22:27], []).append(l)             # one entry per copy of the ligand (residue number + insertion code)
    if len(copies) > 1:                                               # several copies in the chain: use the one in the active site
        fe = [l for l in het if l[12:16].strip().upper() == 'FE']
        xyz_ = lambda l: np.array([float(l[30:38]), float(l[38:46]), float(l[46:54])])
        if fe: pick = min(copies, key=lambda k: min(np.linalg.norm(xyz_(x) - xyz_(fe[0])) for x in copies[k]))
        else: pick = sorted(copies)[0]
        print(f'  {name}: {len(copies)} copies of {code} in chain {lchain}; using residue {pick.strip()}', '(closest to the heme iron)' if fe else '(first listed)')
        lig = copies[pick]
    if code and not lig: raise RuntimeError(f'ligand {code} chain {lchain} not found')
    out = REC / name
    # box
    if lig:
        (REC / f'ligand_{code}_crystal.pdb').write_text(''.join(lig) + 'END\n')
        xyz = np.array([[float(l[30:38]), float(l[38:46]), float(l[46:54])] for l in lig])
        center = xyz.mean(0)
        size = np.array(s['box'], float) if 'box' in s else np.clip(xyz.max(0) - xyz.min(0) + 16, 20, 30)
    elif s.get('center_on_fe'):
        fe = [l for l in het if l[12:16].strip().upper() == 'FE']
        if not fe: raise RuntimeError('no heme iron found (keep_het HEM)')
        center = np.array([float(fe[0][30:38]), float(fe[0][38:46]), float(fe[0][46:54])]); size = np.array([24.0] * 3)
    else: raise RuntimeError('need a ligand or center_on_fe')
    # protein only -> hydrogens at pH 7.4 (missing loops deliberately NOT rebuilt)
    Path(f'{out}_raw.pdb').write_text(''.join(prot) + 'END\n')
    fx = PDBFixer(filename=f'{out}_raw.pdb')
    fx.findMissingResidues(); fx.missingResidues = {}
    fx.findNonstandardResidues(); fx.replaceNonstandardResidues()
    fx.findMissingAtoms(); fx.addMissingAtoms(); fx.addMissingHydrogens(7.4)
    with open(f'{out}_H.pdb', 'w') as fh: PDBFile.writeFile(fx.topology, fx.positions, fh, keepIds=True)

    def run_meeko():
        cmd = [sys.executable, '-m', 'meeko.cli.mk_prepare_receptor', '--read_pdb', f'{out}_H.pdb', '-o', str(out), '-p', '-v',
               '--box_center', *[f'{v:.3f}' for v in center], '--box_size', *[f'{v:.1f}' for v in size]]
        r = subprocess.run(cmd, capture_output=True, text=True)
        if r.returncode != 0 or not Path(f'{out}.pdbqt').exists(): raise RuntimeError('Meeko: ' + r.stderr.strip()[-300:])
    def run_obabel():
        body = [l for l in open(f'{out}_H.pdb') if l.startswith(('ATOM', 'HETATM', 'TER'))]
        Path(f'{out}_H_het.pdb').write_text(''.join(body) + ''.join(het) + 'END\n')
        obabel_rigid(f'{out}_H_het.pdb', f'{out}.pdbqt')
    order = [run_meeko, run_obabel] if s['engine'] == 'meeko' else [run_obabel, run_meeko]
    used, err = None, []
    for fn in order:
        if fn is run_meeko and keep: continue                  # Meeko cannot parametrise a heme: never use it for CYPs
        try: fn(); used = fn.__name__[4:]; break
        except Exception as e: err.append(f'{fn.__name__}: {e}')
    if not used: raise RuntimeError(' | '.join(err))
    Path(f'{out}_vina.conf').write_text(
        f'receptor = {out}.pdbqt\ncenter_x = {center[0]:.3f}\ncenter_y = {center[1]:.3f}\ncenter_z = {center[2]:.3f}\n'
        f'size_x = {size[0]:.1f}\nsize_y = {size[1]:.1f}\nsize_z = {size[2]:.1f}\nnum_modes = 9\nenergy_range = 4\nseed = 2026\n')
    return dict(receptor=name, pdb=s['pdb'], status='OK', engine=used, box_center=' '.join(f'{v:.1f}' for v in center),
                box_size=' '.join(f'{v:.0f}' for v in size))

rows = []
for name, spec in RECEPTORS.items():
    sig = json.dumps(spec, sort_keys=True); sigfile = REC / f'{name}_spec.json'
    if (REC / f'{name}.pdbqt').exists() and (REC / f'{name}_vina.conf').exists() and sigfile.exists() and sigfile.read_text() == sig:
        rows.append(dict(receptor=name, pdb=spec['pdb'], status='OK (already prepared)')); continue
    for f in REC.glob(f'{name}*'): f.unlink()                        # settings changed (or no record): drop the old files and any old docking poses
    for base_ in (WORK, OUT): shutil.rmtree(base_ / 'results' / name, ignore_errors=True)
    try: rows.append(prep_receptor(name, spec)); sigfile.write_text(sig)
    except Exception as e: rows.append(dict(receptor=name, pdb=spec['pdb'], status='FAILED: ' + str(e)[:300]))
RECEPTOR_TABLE = pd.DataFrame(rows)
pd.set_option('display.max_colwidth', 120); print(RECEPTOR_TABLE.to_string(index=False))
READY = {r['receptor'] for r in rows if str(r['status']).startswith('OK')}
if 'dpp4' not in READY: raise SystemExit('The DPP-4 receptor failed: fix it before going on (message above).')
sync(); status('receptors')

## STEP 3: Ligands (RDKit 3D + MMFF, then Meeko PDBQT)

In [ ]:
# STEP 3: ligand preparation
from rdkit import Chem, RDLogger
from rdkit.Chem import AllChem
from meeko import MoleculePreparation, PDBQTWriterLegacy, PDBQTMolecule, RDKitMolCreate
RDLogger.DisableLog('rdApp.*')

def find_file(name):
    p = PROJECT / name
    if p.exists(): return p
    hits = list(PROJECT.rglob(name)); return hits[0] if hits else None

def first_col(df, options):
    low = {c.lower(): c for c in df.columns}
    for o in options:
        if o.lower() in low: return low[o.lower()]
    return None

src = find_file('compounds_passed.csv') or find_file('compounds_scored.csv')
if src is None: raise SystemExit('compounds_passed.csv / compounds_scored.csv not found in the project folder')
df = pd.read_csv(src); print('Loaded', src.name, df.shape)
idc = first_col(df, ['compound_id', 'id', 'name'])
smc = first_col(df, ['smiles_canonical', 'canonical_smiles', 'smiles'])
if smc is None:                                            # compounds_scored.csv has no SMILES: take them from the library file
    for lib in PROJECT.rglob('*.csv'):
        try: ld = pd.read_csv(lib, nrows=5)
        except Exception: continue
        if first_col(ld, ['smiles_canonical', 'canonical_smiles', 'smiles']) and first_col(ld, ['compound_id']) and lib != src:
            full = pd.read_csv(lib); sc = first_col(full, ['smiles_canonical', 'canonical_smiles', 'smiles'])
            keep_cols = [c for c in [first_col(full, ['compound_id']), sc, first_col(full, ['inchikey'])] if c]
            df = df.merge(full[keep_cols], left_on=idc, right_on=first_col(full, ['compound_id']), how='left', suffixes=('', '_lib'))
            smc = sc if sc in df.columns else sc + '_lib'; print('SMILES taken from', lib.name); break
if smc is None: raise SystemExit('No SMILES column found')
pf = first_col(df, ['pass_flag'])
if pf: df = df[df[pf].astype(str).str.lower().isin(['true', '1', 'yes'])].copy()
df = df.rename(columns={idc: 'compound_id', smc: 'smiles'})
ac = first_col(df, ['admet_score'])
if ac: df = df.sort_values(ac, ascending=False)            # best ADMET first: if time runs out, the best candidates are docked first
n_in = len(df)
def ik14(s):
    m = Chem.MolFromSmiles(s); return Chem.MolToInchiKey(m)[:14] if m else s
ikc = first_col(df, ['inchikey'])
df['ik14'] = df[ikc].astype(str).str[:14] if ikc else df.smiles.map(ik14)
if DEDUP_STEREOISOMERS: df = df.drop_duplicates('ik14')
df = df.drop_duplicates('compound_id')
n_unique = len(df)
if MAX_LIGANDS: df = df.head(MAX_LIGANDS)
print(f'compounds in file: {n_in} | after removing duplicate parent structures: {n_unique} | to prepare: {len(df)}')

try:
    from dimorphite_dl import protonate_smiles
except Exception: protonate_smiles = None

LIG = WORK / 'ligands'; prep = MoleculePreparation(); log = []
for _, r in df.iterrows():
    cid, smi, status_ = r.compound_id, r.smiles, 'ok'
    if (LIG / f'{cid}.pdbqt').exists():
        log.append(dict(compound_id=cid, smiles_used=smi, status='ok (existing)')); continue
    try:
        if protonate_smiles:                                # charge state at pH 7.4
            try:
                v = protonate_smiles(smi, ph_min=7.4, ph_max=7.4, max_variants=1)
                if v and isinstance(v[0], str): smi = v[0]
            except Exception: pass
        m = Chem.MolFromSmiles(smi)
        if m is None: raise ValueError('RDKit cannot read SMILES')
        m = Chem.AddHs(m)
        ps = AllChem.ETKDGv3(); ps.randomSeed = 2026
        if AllChem.EmbedMolecule(m, ps) != 0:
            ps.useRandomCoords = True
            if AllChem.EmbedMolecule(m, ps) != 0: raise ValueError('3D embedding failed')
        AllChem.MMFFOptimizeMolecule(m, maxIters=2000)
        txt, ok, err = PDBQTWriterLegacy.write_string(prep.prepare(m)[0])
        if not ok: raise ValueError(err)
        (LIG / f'{cid}.pdbqt').write_text(txt)
    except Exception as e: status_ = f'FAILED: {e}'
    log.append(dict(compound_id=cid, smiles_used=smi, status=status_))
LOG = pd.DataFrame(log); LOG.to_csv(WORK / 'results' / 'ligand_prep_log.csv', index=False)
LIGAND_IDS = [c for c in LOG[LOG.status.str.startswith('ok')].compound_id if (LIG / f'{c}.pdbqt').exists()]
print(f'ligands ready: {len(LIGAND_IDS)} | failed: {(~LOG.status.str.startswith("ok")).sum()}')
for _, r in LOG[~LOG.status.str.startswith('ok')].iterrows(): print('  FAILED', r.compound_id, r.status)
sync(); status('ligands')

## STEP 4: Redocking check (must be under 2.0 Å)
DPP-4 is the gate: if it fails, fix it before Step 5. hERG, CYP2D6 and CYP3A4 (2V0M, ketoconazole) are checked the same way. A failure on a safety receptor only lowers the confidence in its scores and is noted in the report.

The crystal ligand is docked back from a fresh random 3D shape (not the crystal pose, which would be cheating) and compared with the crystal position using spyrmsd, which handles symmetric atoms correctly.

In [ ]:
# STEP 4: redocking validation (DPP-4 gate first, then the safety receptors that have a crystal ligand)
from spyrmsd import rmsd as srmsd
from spyrmsd.molecule import Molecule
from rdkit.Chem import rdMolAlign

REDOCK_SMILES = {   # each crystal ligand, charged as at pH 7.4
    '605': '[NH3+][C@H](Cc1cc(F)ccc1F)c1ccc(cc1)-c1cccc(c1)C(N)=O',                         # DPP-4
    'XB7': 'COc1ccc(CC[NH+]2CCC(CC2)Nc2nc3ccccc3n2Cc2ccc(F)cc2)cc1',                         # hERG
    'PN0': 'CC1([C@@H](N(CCS1)S(=O)(=O)c2ccc(cc2)Oc3ccncc3)C(=O)NO)C',                       # CYP2D6
    'KLN': 'CC(=O)N1CCN(CC1)c1ccc(OC[C@H]2CO[C@](Cn3ccnc3)(O2)c2ccc(Cl)cc2Cl)cc1',            # CYP3A4, ketoconazole
}
REDOCK_EXH = 16
VAL = WORK / 'validation'

def vina_run(conf, lig, out, exh, cpu=1, seed=2026):
    tmp = str(out) + '.part'
    r = subprocess.run([VINA, '--config', str(conf), '--ligand', str(lig), '--out', tmp, '--cpu', str(cpu),
                        '--seed', str(seed), '--exhaustiveness', str(exh)], capture_output=True, text=True)
    if r.returncode != 0 or not os.path.exists(tmp): return False, r.stderr.strip()[-200:]
    os.replace(tmp, out); return True, ''

def best_score(path):
    for line in open(path):
        if line.startswith('REMARK VINA RESULT'): return float(line.split()[3])
    return None

def redock(rname, quiet=False):
    """dock the crystal ligand back into its own receptor; returns dict(verdict, top1, best, ref_score)"""
    code = RECEPTORS[rname]['ligand']
    xtal = REC / f'ligand_{code}_crystal.pdb'
    tmpl = Chem.MolFromSmiles(REDOCK_SMILES[code])
    ref = Chem.RemoveHs(AllChem.AssignBondOrdersFromTemplate(tmpl, Chem.MolFromPDBFile(str(xtal), removeHs=True, sanitize=False)))
    mol = Chem.AddHs(Chem.MolFromSmiles(REDOCK_SMILES[code]))           # fresh 3D shape from SMILES, NOT the crystal pose
    if AllChem.EmbedMolecule(mol, AllChem.ETKDGv3()) != 0: raise RuntimeError('3D embedding failed')
    AllChem.MMFFOptimizeMolecule(mol, maxIters=2000)
    txt, ok, err = PDBQTWriterLegacy.write_string(MoleculePreparation().prepare(mol)[0])
    if not ok: raise RuntimeError(err)
    (VAL / f'{rname}_{code}_input.pdbqt').write_text(txt)
    pose_file = VAL / f'{rname}_{code}_redock_poses.pdbqt'
    t = time.time()
    ok, err = vina_run(REC / f'{rname}_vina.conf', VAL / f'{rname}_{code}_input.pdbqt', pose_file, REDOCK_EXH, cpu=WORKERS)
    if not ok: raise RuntimeError('Vina failed: ' + err)
    scores = [float(l.split()[3]) for l in open(pose_file) if l.startswith('REMARK VINA RESULT')]
    poses = Chem.RemoveHs(RDKitMolCreate.from_pdbqt_mol(PDBQTMolecule.from_file(str(pose_file), skip_typing=True))[0])
    refm = Molecule.from_rdkit(ref); pm0 = Molecule.from_rdkit(poses)
    rm = [srmsd.symmrmsd(refm.coordinates, poses.GetConformer(c.GetId()).GetPositions(), refm.atomicnums, pm0.atomicnums,
                         refm.adjacency_matrix, pm0.adjacency_matrix) for c in poses.GetConformers()]
    print(f'{rname}: redocking {code} took {time.time() - t:.0f} s')
    print(f"{'pose':>4} {'Vina kcal/mol':>14} {'RMSD (A)':>9}")
    for i, (s_, r_) in enumerate(zip(scores, rm), 1): print(f'{i:>4} {s_:>14.2f} {r_:>9.2f}')
    top1, best = float(rm[0]), float(min(rm))
    # overlay files: crystal pose + redocked pose(s) together, to look at in PyMOL / Chimera / Discovery Studio
    def lig_lines(m, conf, resn, ch):
        out_ = []
        for l in Chem.MolToPDBBlock(m, confId=conf).splitlines():
            if l.startswith(('ATOM', 'HETATM')): out_.append('HETATM' + l[6:17] + resn[:3].ljust(3) + ' ' + ch + '   1' + l[26:] + '\n')
        return out_
    cids = [c.GetId() for c in poses.GetConformers()]; ibest = int(np.argmin(rm))
    sets = [('XTL', 'X', ref, ref.GetConformer().GetId(), 'crystal pose'), ('TOP', 'Z', poses, cids[0], f'redocked pose 1 ({top1:.2f} A)')]
    if ibest != 0: sets.append(('BST', 'Y', poses, cids[ibest], f'redocked pose {ibest + 1} ({best:.2f} A)'))
    base = VAL / f'{rname}_{code}_overlay'
    with open(f'{base}_ligands.pdb', 'w') as fh:                       # ligands only, one MODEL each
        for n_, (rn, ch, m, cf, note) in enumerate(sets, 1):
            fh.write(f'MODEL {n_}\nREMARK {note}\n' + ''.join(lig_lines(m, cf, rn, ch)) + 'ENDMDL\n')
        fh.write('END\n')
    with open(f'{base}_complex.pdb', 'w') as fh:                       # protein + crystal (chain X) + redocked (chain Z, and Y if a lower pose was closer)
        fh.write(''.join(l for l in open(REC / f'{rname}_H.pdb') if l.startswith('ATOM')) + 'TER\n')
        for rn, ch, m, cf, note in sets: fh.write(''.join(lig_lines(m, cf, rn, ch)))
        fh.write('END\n')
    verdict = 'PASS' if top1 < 2.0 else ('PARTIAL' if best < 2.0 else 'FAIL')
    pd.DataFrame(dict(pose=range(1, len(rm) + 1), vina_kcal=scores, rmsd_A=[round(float(x), 2) for x in rm])).to_csv(VAL / f'redock_rmsd_{rname}.csv', index=False)
    print(f'REDOCK {verdict} ({rname}): top-ranked pose {top1:.2f} A | best of {len(rm)} poses {best:.2f} A | reference score {scores[0]:.2f} kcal/mol\n')
    print(f'  overlay files: validation/{rname}_{code}_overlay_complex.pdb (protein + both poses) and _overlay_ligands.pdb')
    return dict(receptor=rname, ligand=code, verdict=verdict, top1_rmsd=round(top1, 2), best_rmsd=round(best, 2), ref_score=scores[0])

# --- gate 1: DPP-4 (this decides whether Step 5 can be trusted) ---
d = redock('dpp4')
REDOCK_VERDICT, REF_SCORE, code = d['verdict'], d['ref_score'], RECEPTORS['dpp4']['ligand']
if REDOCK_VERDICT == 'PASS': print('The DPP-4 box and receptor setup reproduce the crystal pose. Safe to continue.')
elif REDOCK_VERDICT == 'PARTIAL': print('The right pose is found but not ranked first. Usable: say so in the methods.')
else: print('WARNING: FAIL. Do not trust the screen yet. Check the box (receptors/dpp4.box.pdb), the ligand charge and the chain choice before Step 5.')
shutil.copy(VAL / 'redock_rmsd_dpp4.csv', VAL / 'redock_rmsd.csv')

# --- safety receptors: a check on how far to trust their scores. A failure here never stops the run. ---
REDOCK_ROWS = [d]
for rname, spec in RECEPTORS.items():
    if rname == 'dpp4': continue
    if rname not in READY: REDOCK_ROWS.append(dict(receptor=rname, verdict='n/a (receptor not prepared)')); continue
    if not spec.get('ligand') or spec.get('ligand') not in REDOCK_SMILES:
        REDOCK_ROWS.append(dict(receptor=rname, verdict='n/a (no crystal ligand in this entry)')); continue
    if BUDGET.remaining() - BUDGET.margin < 30:
        REDOCK_ROWS.append(dict(receptor=rname, verdict='skipped (time budget)')); continue
    try: REDOCK_ROWS.append(redock(rname))
    except Exception as e: REDOCK_ROWS.append(dict(receptor=rname, verdict='error: ' + str(e)[:120]))
REDOCK_TABLE = pd.DataFrame(REDOCK_ROWS)
REDOCK_TABLE.to_csv(VAL / 'redock_summary.csv', index=False)
print(REDOCK_TABLE.to_string(index=False))
bad = [r['receptor'] for r in REDOCK_ROWS[1:] if r['verdict'] not in ('PASS', 'PARTIAL') and not r['verdict'].startswith(('n/a', 'skipped'))]
if bad: print('\nLow-confidence safety receptors (their scores are flags only, and this is noted in the report):', ', '.join(bad))
sync(); status('redock')

## STEP 5: Docking
5A times a few compounds and picks the search effort that fits your time budget. 5B docks the whole library. 5C docks the top hits against the three safety receptors.

In [ ]:
# STEP 5A: timing test and settings
def parse_pose_ok(p):
    return os.path.exists(p) and best_score(p) is not None

def dock_many(conf, ids, outdir, exh, label, cpu_workers=None, reserve_min=0):
    """docks ids one CPU each, resumable, stops cleanly when the time budget runs low. returns {id: (score, pose_path)}"""
    outdir = Path(outdir); poses = outdir / 'poses'; poses.mkdir(parents=True, exist_ok=True)
    workers = cpu_workers or WORKERS
    stop = threading.Event(); res = {}
    def job(cid):
        pose = poses / f'{cid}_out.pdbqt'
        if parse_pose_ok(pose): return cid, best_score(pose), str(pose), 'cached'
        if stop.is_set(): return cid, None, None, 'skipped'
        ok, err = vina_run(conf, LIG / f'{cid}.pdbqt', pose, exh)
        return (cid, best_score(pose), str(pose), '') if ok else (cid, None, None, err)
    t0, done_n, fails, skipped = time.time(), 0, [], 0
    with ThreadPoolExecutor(workers) as ex:
        futs = [ex.submit(job, c) for c in ids]
        for f in as_completed(futs):
            cid, sc, pf, note = f.result(); done_n += 1
            if sc is not None: res[cid] = (sc, pf)
            elif note == 'skipped': skipped += 1
            else: fails.append((cid, note))
            if BUDGET.remaining() <= BUDGET.margin + reserve_min and not stop.is_set():
                stop.set(); print(f'\n*** time limit reached ({label}): finishing running jobs and saving ***')
            if done_n % 10 == 0 or done_n == len(ids):
                rate = done_n / max(1e-9, time.time() - t0)
                print(f'{label}: {done_n}/{len(ids)} | ETA {(len(ids) - done_n) / max(rate, 1e-9) / 60:.0f} min | budget left {BUDGET.remaining():.0f} min', flush=True)
            if done_n % SYNC_EVERY == 0: sync(['results'])
    if fails: print('failed:', fails[:5], '...' if len(fails) > 5 else '')
    if skipped: print(f'NOT docked because the time budget ran out: {skipped} compounds (re-run this cell later to continue; finished ones are kept)')
    return res

DPP4_CONF = REC / 'dpp4_vina.conf'
exh = EXHAUSTIVENESS
if not exh:
    pilot = WORK / '_pilot'; shutil.rmtree(pilot, ignore_errors=True); pilot.mkdir()
    sel = LIGAND_IDS[:max(1, min(WORKERS, len(LIGAND_IDS)))]
    t = time.time()
    with ThreadPoolExecutor(len(sel)) as ex: list(ex.map(lambda c: vina_run(DPP4_CONF, LIG / f'{c}.pdbqt', pilot / f'{c}.pdbqt', 8), sel))
    cost8 = max(time.time() - t, 1.0)       # one wave = every core busy with one ligand, so wall seconds = core-seconds per ligand at exhaustiveness 8
    n_anti = TOP_N * (len(READY) - 1)
    ANTI_RESERVE_MIN = n_anti * cost8 * (ANTI_EXHAUSTIVENESS / 8) * 1.5 / WORKERS / 60 + 2     # time kept free for the safety docking
    avail = (BUDGET.remaining() - BUDGET.margin - ANTI_RESERVE_MIN) * 60 * 0.85 - 120
    exh, proj = 4, None
    for e in (16, 8, 4):
        proj = len(LIGAND_IDS) * cost8 * (e / 8) / WORKERS
        if proj <= avail: exh = e; break
    print(f'timing test: about {cost8:.0f} s per compound per core at exhaustiveness 8')
    print(f'-> using exhaustiveness {exh}: projected {len(LIGAND_IDS) * cost8 * (exh / 8) / WORKERS / 60:.0f} min for {len(LIGAND_IDS)} compounds on {WORKERS} cores')
    if len(LIGAND_IDS) * cost8 * (exh / 8) / WORKERS > avail:
        print('WARNING: even exhaustiveness 4 will not finish inside your time budget. Docking will stop at the limit; the best-ADMET compounds are docked first.')
else:
    print('using your exhaustiveness', exh)
    ANTI_RESERVE_MIN = 12
EXH_USED = exh

In [ ]:
# STEP 5B: therapeutic docking, DPP-4 (all prepared ligands)
if REDOCK_VERDICT == 'FAIL': print('REMINDER: the redocking check failed; treat these scores as provisional.')
res = dock_many(DPP4_CONF, LIGAND_IDS, WORK / 'results' / 'target', EXH_USED, 'DPP-4', reserve_min=ANTI_RESERVE_MIN)

def heavy_atoms(path):
    return sum(1 for l in open(path) if l.startswith(('ATOM', 'HETATM')) and l.split()[-1] not in ('HD', 'H', 'HS'))
rows = []
for cid, (sc, pf) in res.items():
    ha = heavy_atoms(LIG / f'{cid}.pdbqt')
    rows.append(dict(compound_id=cid, vina_score_kcal=sc, pose_file=os.path.relpath(pf, WORK), heavy_atoms=ha, ligand_efficiency=round(-sc / ha, 3)))
TARGET = pd.DataFrame(rows).sort_values('vina_score_kcal').reset_index(drop=True)
TARGET.to_csv(WORK / 'results' / 'docking_target.csv', index=False)
print(f'\ndocked {len(TARGET)} of {len(LIGAND_IDS)} | file: results/docking_target.csv')
print(f'reference (redocked {code}) = {REF_SCORE:.2f} kcal/mol | compounds within 1 kcal/mol of it: {(TARGET.vina_score_kcal <= REF_SCORE + 1).sum()}')
print(TARGET.head(TOP_N).to_string(index=False))
sync(); status('target docking')

In [ ]:
# STEP 5C: inverse (safety) docking of the top hits
asc = RANK_BY == 'vina_score_kcal'
TOP = TARGET.sort_values(RANK_BY, ascending=asc).head(TOP_N)
top_ids = list(TOP.compound_id)
print('Top hits by', RANK_BY, ':', top_ids)
anti_rows = []
for rname, label in (('herg', 'hERG'), ('cyp3a4', 'CYP3A4'), ('cyp2d6', 'CYP2D6')):
    if rname not in READY:
        print(f'{label}: receptor not available (see Step 2 table), skipped'); continue
    exh_a = ANTI_EXHAUSTIVENESS if BUDGET.remaining() - BUDGET.margin > 15 else 8
    r_ = dock_many(REC / f'{rname}_vina.conf', top_ids, WORK / 'results' / rname, exh_a, label)
    for cid, (sc, pf) in r_.items(): anti_rows.append(dict(compound_id=cid, antitarget=label, vina_score_kcal=sc, pose_file=os.path.relpath(pf, WORK)))
ANTI = pd.DataFrame(anti_rows, columns=['compound_id', 'antitarget', 'vina_score_kcal', 'pose_file'])
ANTI.to_csv(WORK / 'results' / 'docking_antitargets.csv', index=False)
if len(ANTI):
    tgt = TARGET.set_index('compound_id').vina_score_kcal
    ANTI['target_score_kcal'] = ANTI.compound_id.map(tgt)
    ANTI['delta_vs_target_kcal'] = (ANTI.vina_score_kcal - ANTI.target_score_kcal).round(2)   # negative = binds the safety receptor MORE tightly than DPP-4
    ANTI['stronger_than_target'] = ANTI.delta_vs_target_kcal < 0
    ANTI.to_csv(WORK / 'results' / 'docking_antitargets.csv', index=False)
    print(ANTI.pivot(index='compound_id', columns='antitarget', values='vina_score_kcal').reindex(top_ids).to_string())
    print('\nNote: hERG docking is a flag, not a verdict. The pore is large and flexible; CYP2D6/3A4 scores are only as good as the heme set-up (check the Step 2 table).')
sync(); status('safety docking')

## STEP 6: Interactions (PLIP) and the summary table

In [ ]:
# STEP 6: PLIP on the top hits + final report
def pose_to_ligand_lines(pose_pdbqt):
    """best pose as PDB lines (with hydrogens) named LIG, chain Z"""
    pm = PDBQTMolecule.from_file(str(pose_pdbqt), skip_typing=True)
    m = RDKitMolCreate.from_pdbqt_mol(pm)[0]
    blk = Chem.MolToPDBBlock(m, confId=m.GetConformers().__iter__().__next__().GetId())
    out = []
    for l in blk.splitlines():
        if l.startswith(('ATOM', 'HETATM')):
            l = 'HETATM' + l[6:].ljust(74)
            l = l[:17] + 'LIG' + ' ' + 'Z' + '   1' + l[26:]
            out.append(l.rstrip() + '\n')
        elif l.startswith('CONECT'): out.append(l + '\n')
    return out

def lab(i): return f'{i.restype}{i.resnr}({i.reschain})'
def plip_profile(cid, pose_rel):
    from plip.structure.preparation import PDBComplex
    prot = [l for l in open(REC / 'dpp4_H.pdb') if l.startswith('ATOM')]
    cpx = WORK / 'results' / 'plip' / f'{cid}_complex.pdb'; cpx.parent.mkdir(parents=True, exist_ok=True)
    cpx.write_text(''.join(prot) + 'TER\n' + ''.join(pose_to_ligand_lines(WORK / pose_rel)) + 'END\n')
    c = PDBComplex(); c.load_pdb(str(cpx)); c.analyze()
    d = dict(hbond_residues='', hydrophobic_residues='', salt_bridge_residues='', pi_residues='', n_hbonds=0, n_hydrophobic=0)
    for key, s in c.interaction_sets.items():
        if not key.startswith('LIG'): continue
        hb = list(s.hbonds_ldon) + list(s.hbonds_pdon)
        hp = list(s.hydrophobic_contacts)
        sb = list(s.saltbridge_lneg) + list(s.saltbridge_pneg)
        pi = list(s.pistacking) + list(s.pication_laro) + list(s.pication_paro)
        uniq = lambda xs: '; '.join(sorted({lab(x) for x in xs}))
        d.update(hbond_residues=uniq(hb), hydrophobic_residues=uniq(hp), salt_bridge_residues=uniq(sb), pi_residues=uniq(pi),
                 n_hbonds=len(hb), n_hydrophobic=len(hp))
    return d

summary = TOP.copy().reset_index(drop=True)
summary.insert(0, 'rank', range(1, len(summary) + 1))
if len(ANTI):
    wide = ANTI.pivot(index='compound_id', columns='antitarget', values='vina_score_kcal').add_suffix('_score_kcal')
    summary = summary.merge(wide, left_on='compound_id', right_index=True, how='left')
    worst = ANTI.groupby('compound_id').delta_vs_target_kcal.min().rename('worst_delta_vs_target_kcal')
    summary = summary.merge(worst, left_on='compound_id', right_index=True, how='left')
try:
    prof = []
    for _, r in summary.iterrows():
        try: prof.append(plip_profile(r.compound_id, r.pose_file))
        except Exception as e: prof.append(dict(hbond_residues='PLIP failed: ' + str(e)[:80]))
    summary = pd.concat([summary, pd.DataFrame(prof)], axis=1)
except ImportError:
    print('PLIP not installed: report written without interaction columns')
LAB = {'herg': 'hERG', 'cyp3a4': 'CYP3A4', 'cyp2d6': 'CYP2D6'}
summary['safety_redock_check'] = '; '.join(f"{LAB[r['receptor']]}: {r['verdict']}" for r in REDOCK_ROWS[1:] if r['receptor'] in LAB)
summary['reference_score_kcal'] = REF_SCORE
summary['redock_verdict'] = REDOCK_VERDICT
summary['exhaustiveness'] = EXH_USED
summary.to_csv(WORK / 'results' / 'summary_report.csv', index=False)
summary.to_json(WORK / 'results' / 'summary_report.json', orient='records', indent=2)
show = [c for c in ['rank', 'compound_id', 'vina_score_kcal', 'ligand_efficiency', 'hERG_score_kcal', 'CYP3A4_score_kcal', 'CYP2D6_score_kcal', 'hbond_residues'] if c in summary.columns]
print(summary[show].to_string(index=False))
top1_row = summary.iloc[0]
print(f"\nBest compound: {top1_row.compound_id} | {top1_row.vina_score_kcal} kcal/mol")
print('  H-bonds    :', top1_row.get('hbond_residues', ''))
print('  Hydrophobic:', top1_row.get('hydrophobic_residues', ''))

sync(); status('done')
print('\nSaved in Drive:', OUT)
for p in sorted((OUT / 'results').glob('*.csv')) + sorted((OUT / 'results').glob('*.json')): print('  ', p.relative_to(PROJECT))